<a href="https://colab.research.google.com/github/tusharj23/ProteinBERT/blob/main/TAPE_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tape-proteins

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 1.3 MB/s eta 0:00:00


In [2]:
!wget https://s3.amazonaws.com/songlabdata/proteindata/data_pytorch/secondary_structure.tar.gz

--2026-03-11 16:38:10--  https://s3.amazonaws.com/songlabdata/proteindata/data_pytorch/secondary_structure.tar.gz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 3.5.16.219, 16.15.207.225, 16.15.191.23, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|3.5.16.219|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 251794897 (240M) [application/x-tar]
Saving to: ‘secondary_structure.tar.gz’

secondary_structure 100%[===================>] 240.13M  21.9MB/s    in 11s     

2026-03-11 16:38:21 (22.7 MB/s) - ‘secondary_structure.tar.gz’ saved [251794897/251794897]



In [3]:
!mkdir -p data
!tar -xzf secondary_structure.tar.gz -C data

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import math
from tape.datasets import SecondaryStructureDataset

# Hyperparameters

MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 5
LR = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Dataset Wrapper

class TapeDataset(Dataset):

    def __init__(self, split):

        self.dataset = SecondaryStructureDataset(
            data_path="./data",
            split=split
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        token_ids, input_mask, labels = self.dataset[idx]

        seq = torch.tensor(token_ids)
        label = torch.tensor(labels)

        seq = seq[:MAX_LEN]
        label = label[:MAX_LEN]

        if len(seq) < MAX_LEN:

            pad_len = MAX_LEN - len(seq)

            seq = torch.cat([seq, torch.zeros(pad_len).long()])
            label = torch.cat([label, torch.full((pad_len,), -1).long()])

        return seq, label


# Load Dataset

train_dataset = TapeDataset("train")
valid_dataset = TapeDataset("valid")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)


# Detect Vocabulary Size

max_token = 0

for i in range(500):

    token_ids, _, _ = train_dataset.dataset[i]

    max_token = max(max_token, max(token_ids))

VOCAB_SIZE = max_token + 1

print("Detected vocab size:", VOCAB_SIZE)


# Positional Encoding

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=1000):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]


# Transformer Model

class ProteinTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=128,
        n_heads=4,
        n_layers=3,
        ff_dim=256,
        num_classes=3
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.pos_encoding = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):

        x = self.embedding(x)

        x = self.pos_encoding(x)

        x = self.transformer(x)

        x = self.fc(x)

        return x


# Initialize Model

model = ProteinTransformer(VOCAB_SIZE).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-1)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)



# Training Loop

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for seq, label in train_loader:

        seq = seq.to(device)
        label = label.to(device)

        optimizer.zero_grad()

        output = model(seq)

        output = output.view(-1, 3)
        label = label.view(-1)

        loss = criterion(output, label)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")


# Evaluation

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for seq, label in valid_loader:

        seq = seq.to(device)
        label = label.to(device)

        output = model(seq)

        pred = torch.argmax(output, dim=-1)

        mask = label != -1

        correct += ((pred == label) * mask).sum().item()
        total += mask.sum().item()

accuracy = correct / total

print("Validation Accuracy:", accuracy)

Detected vocab size: 29
Epoch 1/5 | Loss: 0.9749
Epoch 2/5 | Loss: 0.9544
Epoch 3/5 | Loss: 0.9493
Epoch 4/5 | Loss: 0.9428
Epoch 5/5 | Loss: 0.9355
Validation Accuracy: 0.5525678465452459


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import math
from tape.datasets import SecondaryStructureDataset

# Hyperparameters


# MAX_LEN = 256
# BATCH_SIZE = 8
# EPOCHS = 10
LR = 1e-4
MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 10
d_model = 256
heads = 8
layers = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Dataset Wrapper


class TapeDataset(Dataset):

    def __init__(self, split):

        self.dataset = SecondaryStructureDataset(
            data_path="./data",
            split=split
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        token_ids, input_mask, labels = self.dataset[idx]

        seq = torch.tensor(token_ids)
        label = torch.tensor(labels)

        seq = seq[:MAX_LEN]
        label = label[:MAX_LEN]

        if len(seq) < MAX_LEN:

            pad_len = MAX_LEN - len(seq)

            seq = torch.cat([seq, torch.zeros(pad_len).long()])
            label = torch.cat([label, torch.full((pad_len,), -1).long()])

        return seq, label



# Load Dataset


train_dataset = TapeDataset("train")
valid_dataset = TapeDataset("valid")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)



# Detect Vocabulary Size


max_token = 0

for i in range(500):
    token_ids, _, _ = train_dataset.dataset[i]
    max_token = max(max_token, max(token_ids))

VOCAB_SIZE = max_token + 1

print("Detected vocab size:", VOCAB_SIZE)



# Positional Encoding


class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=1000):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]



# Transformer Model


class ProteinTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=256,
        n_heads=8,
        n_layers=4,
        ff_dim=512,
        num_classes=3
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.norm = nn.LayerNorm(d_model)

        self.pos_encoding = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):

        # padding mask
        mask = (x == 0)

        x = self.embedding(x)

        x = self.norm(x)

        x = self.pos_encoding(x)

        x = self.transformer(x, src_key_padding_mask=mask)

        x = self.fc(x)

        return x



# Initialize Model


model = ProteinTransformer(VOCAB_SIZE).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-1)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)



# Training Loop


for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for seq, label in train_loader:

        seq = seq.to(device)
        label = label.to(device)

        optimizer.zero_grad()

        output = model(seq)

        output = output.view(-1, 3)
        label = label.view(-1)

        loss = criterion(output, label)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")



# Evaluation


model.eval()

correct = 0
total = 0

with torch.no_grad():

    for seq, label in valid_loader:

        seq = seq.to(device)
        label = label.to(device)

        output = model(seq)

        pred = torch.argmax(output, dim=-1)

        mask = label != -1

        correct += ((pred == label) * mask).sum().item()
        total += mask.sum().item()

accuracy = correct / total

print("Validation Accuracy:", accuracy)

Detected vocab size: 29
Epoch 1/10 | Loss: 0.9756
Epoch 2/10 | Loss: 0.9589
Epoch 3/10 | Loss: 0.9505
Epoch 4/10 | Loss: 0.9437
Epoch 5/10 | Loss: 0.9364
Epoch 6/10 | Loss: 0.9252
Epoch 7/10 | Loss: 0.9006
Epoch 8/10 | Loss: 0.8680
Epoch 9/10 | Loss: 0.8434
Epoch 10/10 | Loss: 0.8257


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Validation Accuracy: 0.6370904263772424
